# Zero-Shot Depth Estimation with DPT and Gradio

Run all cells in a fresh Python3.11 CPU kernel. Downloads the pinned official Intel DPT-Hybrid weights and the exact recovered article image. Predictions are relative inverse depth, not calibrated metric distance. This notebook tests a local Gradio endpoint, not public sharing or deployment. The original code/transcripts remain in the appendix.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
from tempfile import TemporaryDirectory
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check', 'torch==2.10.0', 'torchvision==0.25.0', 'transformers==4.57.6', 'timm==1.0.22', 'gradio==6.3.0', 'gradio_client==2.0.3', 'numpy==2.3.5', 'pillow==12.0.0', 'certifi==2026.10.23', 'ipywidgets==8.1.7'], check=True)
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
import certifi
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()
import numpy as np
import torch
import transformers
from transformers import pipeline, AutoImageProcessor
from PIL import Image
from IPython.display import display
import hashlib
import urllib.request
workspace = TemporaryDirectory(prefix='tdb-dpt-')
work_dir = Path(workspace.name)
os.environ['GRADIO_TEMP_DIR'] = str(work_dir / 'gradio')
torch.set_num_threads(2)
torch.manual_seed(42)
torch.set_printoptions(precision=4, edgeitems=2)
print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__, '| Transformers:', transformers.__version__)

In [ ]:
print('Pinned PyTorch, torchvision, Transformers, timm and Gradio packages installed in the active kernel.')

In [ ]:
MODEL_ID = 'Intel/dpt-hybrid-midas'
MODEL_REVISION = '11eaf7a1cf4bd70740697dbc216f98980c0aeb03'
processor = AutoImageProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION, token=False, use_fast=False)
depth_estimator = pipeline('depth-estimation', model=MODEL_ID, revision=MODEL_REVISION,
                           token=False, device=-1, image_processor=processor,
                           model_kwargs={'use_safetensors': False})
depth_estimator.model.eval()
print('Loaded:', MODEL_ID, '| revision:', MODEL_REVISION)

In [ ]:
IMAGE_URL = 'https://cdn-images-1.medium.com/max/2000/1*wQvvym5RV5bqG4u6vbKi6A.png'
image_path = work_dir / 'source-input.png'
request = urllib.request.Request(IMAGE_URL, headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(request, timeout=60) as response:
    image_path.write_bytes(response.read())
image_sha256 = hashlib.sha256(image_path.read_bytes()).hexdigest()
assert image_sha256 == '143fc1957cf6a313b0821cc6d931a81431c7e1ea59cecaed404784bb524963a5'
raw_image = Image.open(image_path).convert('RGB')
raw_image = raw_image.resize((806, 621))
print('Original image SHA-256:', image_sha256)
print('Input width / height:', raw_image.size)

In [ ]:
output = depth_estimator(raw_image)
predicted_depth = output['predicted_depth']
print('Output keys:', sorted(output))
print('Predicted relative inverse-depth shape:', tuple(predicted_depth.shape))
print('Finite:', bool(torch.isfinite(predicted_depth).all()))
print('Range:', round(predicted_depth.min().item(), 4), round(predicted_depth.max().item(), 4))

In [ ]:
print(output['predicted_depth'].shape)

In [ ]:
print(output['predicted_depth'][None, None].shape)

In [ ]:
prediction = torch.nn.functional.interpolate(
    output['predicted_depth'][None, None],
    size=raw_image.size[::-1], mode='bicubic', align_corners=False,
)
print(prediction.shape)

In [ ]:
print(raw_image.size[::-1])

In [ ]:
print(prediction)

In [ ]:
def normalize_depth(tensor):
    values = tensor.detach().cpu().numpy().squeeze()
    if values.ndim != 2 or not np.isfinite(values).all():
        raise ValueError('Expected a finite two-dimensional depth map.')
    maximum = float(values.max())
    formatted = np.zeros(values.shape, dtype=np.uint8) if maximum <= 0 else np.clip(values * 255 / maximum, 0, 255).astype(np.uint8)
    return Image.fromarray(formatted)

depth = normalize_depth(prediction)
display(depth)
print('Relative-depth visualization:', depth.mode, depth.size)
print('Pixel range:', int(np.asarray(depth).min()), int(np.asarray(depth).max()))

In [ ]:
import gradio as gr
from gradio_client import Client, handle_file
print('Gradio:', gr.__version__)

In [ ]:
def launch(input_image):
    if input_image is None:
        raise ValueError('Upload an image before running depth estimation.')
    input_image = input_image.convert('RGB')
    out = depth_estimator(input_image)
    prediction = torch.nn.functional.interpolate(
        out['predicted_depth'][None, None], size=input_image.size[::-1],
        mode='bicubic', align_corners=False,
    )
    return normalize_depth(prediction)

print('Depth-estimation callback ready.')

In [ ]:
iface = gr.Interface(launch, inputs=gr.Image(type='pil', format='png'), outputs=gr.Image(type='pil', format='png'),
                     api_name='predict_depth')
_, local_url, _ = iface.launch(share=False, server_name='127.0.0.1', server_port=None,
                             prevent_thread_lock=True, quiet=True)
client = Client(local_url, verbose=False)
result_path = client.predict(handle_file(str(image_path)), api_name='/predict_depth')
with Image.open(result_path) as result_image:
    endpoint_mode = result_image.mode
    if endpoint_mode == 'RGB':
        channels = np.asarray(result_image)
        np.testing.assert_array_equal(channels[..., 0], channels[..., 1])
        np.testing.assert_array_equal(channels[..., 1], channels[..., 2])
    endpoint_image = result_image.convert('L')
np.testing.assert_array_equal(np.asarray(endpoint_image), np.asarray(depth))
print('Local Gradio /predict_depth endpoint returned:', endpoint_mode, endpoint_image.size)
print('Endpoint pixels match the direct source-image prediction:', True)

In [ ]:
iface.close()
print('Local Gradio demo closed.')

In [ ]:
assert raw_image.size == (806,621)
assert predicted_depth.shape == (621,806)
assert prediction.shape == (1,1,621,806)
assert not depth_estimator.model.training and all(p.grad is None for p in depth_estimator.model.parameters())
# Independent direct processor/model path and resize, not a second pipeline call.
inputs = depth_estimator.image_processor(images=raw_image, return_tensors='pt')
with torch.inference_mode():
    native_depth = depth_estimator.model(**inputs).predicted_depth
assert native_depth.shape == (1,384,384)
reference = torch.nn.functional.interpolate(native_depth[:,None], size=(621,806), mode='bicubic', align_corners=False)[0,0]
torch.testing.assert_close(predicted_depth, reference, atol=1e-4, rtol=1e-5)
np.testing.assert_allclose(prediction[0,0].numpy(), reference.numpy(), atol=1e-4, rtol=1e-5)
values = reference.numpy()
reference_pixels = np.clip(values * 255 / values.max(),0,255).astype(np.uint8)
np.testing.assert_array_equal(np.asarray(depth),reference_pixels)
np.testing.assert_array_equal(np.asarray(endpoint_image),reference_pixels)
assert np.ptp(np.asarray(depth)) > 100
assert (np.asarray(normalize_depth(torch.zeros(1,1,3,5))) == 0).all()
assert (np.asarray(normalize_depth(torch.ones(1,1,3,5))) == 255).all()
for invalid in [torch.full((2,2),float('nan')),torch.zeros(2,3,4)]:
    try:normalize_depth(invalid)
    except ValueError:pass
    else:raise AssertionError('Invalid map must fail.')
try:launch(None)
except ValueError:pass
else:raise AssertionError('Missing input must fail.')
assert not iface.is_running
print('Actual pinned DPT native inference, independent bicubic resize, full806×621 finite inverse-depth map and all visualization pixels passed.')
print('Real local Gradio upload/endpoint matches every direct prediction pixel; invalid input/constant-map cases and server closure passed.')
workspace.cleanup()
assert not work_dir.exists()
print('Owned input/cache temporary files cleaned up.')

## Original source code and transcripts

Nonexecuting original blocks:

### Original block 1

```text
!pip install transformers
!pip install gradio
!pip install timm
!pip install torchvision
```

### Original block 2

```text
depth_estimator = pipeline(task="depth-estimation",
                        model="./models/Intel/dpt-hybrid-midas")
```

### Original block 3

```text
raw_image = Image.open('palestine demo.jpg')
raw_image.resize((806, 621))
```

### Original block 4

```text
output = depth_estimator(raw_image)
output
```

### Original block 5

```text
{'predicted_depth': tensor([[[ 781.4485,  773.2712,  775.9324,  ...,  214.4251,  212.5703,
            167.9315],
          [ 761.0817,  774.3453,  776.3634,  ...,  214.7691,  211.7175,
            214.0400],
          [ 780.3962,  777.5194,  777.1779,  ...,  215.2407,  213.9116,
            214.5150],
          ...,
          [3135.9419, 3142.6958, 3133.6479,  ..., 2930.2998, 2921.9617,
           2921.4111],
          [3141.9880, 3140.8049, 3141.9360,  ..., 2935.5476, 2939.6582,
           2923.6726],
          [3135.9834, 3143.6619, 3137.9397,  ..., 2964.1797, 2962.6919,
           2956.1938]]]),
 'depth': <PIL.Image.Image image mode=L size=1300x855>}
```

### Original block 6

```text
output["predicted_depth"].shape
```

### Original block 7

```text
output["predicted_depth"].unsqueeze(1).shape
```

### Original block 8

```text
prediction = torch.nn.functional.interpolate(
    output["predicted_depth"].unsqueeze(1),
    size=raw_image.size[::-1],
    mode="bicubic",
    align_corners=False,
)
prediction.shape
```

### Original block 9

```text
raw_image.size[::-1]
```

### Original block 10

```text
prediction
```

### Original block 11

```text
output = prediction.squeeze().numpy()
formatted = (output * 255 / np.max(output)).astype("uint8")
depth = Image.fromarray(formatted)
depth
```

### Original block 12

```text
import os
import gradio as gr
from transformers import pipeline
```

### Original block 13

```text
def launch(input_image):
    out = depth_estimator(input_image)
    # resize the prediction
    prediction = torch.nn.functional.interpolate(
        out["predicted_depth"].unsqueeze(1),
        size=input_image.size[::-1],
        mode="bicubic",
        align_corners=False,
    )
    
    # normalize the prediction
    output = prediction.squeeze().numpy()
    formatted = (output * 255 / np.max(output)).astype("uint8")
    depth = Image.fromarray(formatted)
    return depth
```

### Original block 14

```text
iface = gr.Interface(launch, 
                     inputs=gr.Image(type='pil'), 
                     outputs=gr.Image(type='pil'))
iface.launch(share=True, server_port=int(os.environ['PORT1']))
```

### Original block 15

```text
iface.close()
```

### Original quoted outputs

torch.Size([1, 384, 384])

torch.Size([1, 1, 384, 384])

torch.Size([1, 1, 855, 1300])

((855, 1300),)

tensor([[[[ 784.8216, 784.0563, 781.3138, …, 173.5532, 161.0429, 157.4184], [ 778.5426, 778.2037, 776.9430, …, 182.8467, 173.4256, 170.6889], [ 765.5461, 766.0751, 767.8272, …, 200.2888, 196.7097, 195.6512], …, [3140.3447, 3140.4302, 3140.7307, …, 2938.8877, 2935.1787, 2934.1406], [3136.3120, 3136.7642, 3138.4766, …, 2953.0996, 2951.0381, 2950.4395], [3134.3740, 3135.0039, 3137.3967, …, 2960.8267, 2959.5129, 2959.1138]]]])